# Multi-*e*GO — mg reference simulation with GROMACS

This notebook runs the full **molten-globule (mg) reference simulation** required by the multi-eGO
workflow, using GROMACS.  Starting from a PDB file it carries out every setup step:

| Step | Tool | What it does |
|------|------|--------------|
| 3 | `gmx pdb2gmx` | Build an AMBER topology from the PDB |
| 4 | `gmx editconf` | Define the periodic simulation box |
| 5 | `mego --egos mg` | Generate the mg force-field parameters |
| 7 | `gmx mdrun` | Energy minimisation |
| 8 | `gmx mdrun` | NVT production run |

The resulting trajectory (`run.xtc`) and run-input file (`run.tpr`) are everything you need for
the next step of the multi-eGO workflow (`cmdata` → `make_mat` → `mego`).

> **Before you start:** go to *Runtime → Change runtime type* and select **GPU** to accelerate
> the production run.

---

## Choose your GROMACS installation (run **one** of the two options below)

| | Option A — apt | Option B — compile |
|-|---------------|--------------------|
| Time | ~2 min | ~20 min |
| Version | Ubuntu-packaged | multi-eGO fork (recommended) |
| GPU support | depends on runtime | yes (CUDA auto-detected) |

Run **either** the *Option A* cell **or** the *Option B* cell — not both.

## ▶ Option A — Install GROMACS via apt  *(fast, ~2 min)*

Installs the GROMACS version packaged by Ubuntu.  Covers all steps of the mg workflow.
Skip this cell and run **Option B** instead if you want the multi-eGO GROMACS fork.

In [ ]:
# ── Option A — run this cell OR the Option B cell below, not both ────────────
!apt update -qq && apt install -y -qq gromacs
!gmx --version | head -5

## ▶ Option B — Compile GROMACS from the multi-eGO fork  *(~20 min, recommended)*

Builds the exact version of GROMACS used during multi-eGO development, with CUDA GPU support
when a GPU runtime is selected.  Compilation takes ~20 minutes on Colab CPUs.
Skip this cell if you already ran **Option A** above.

In [ ]:
# ── Option B — run this cell OR the Option A cell above, not both ────────────
import subprocess, os

!apt update -qq && apt install -y -qq cmake build-essential libfftw3-dev ninja-build

# Clone the multi-eGO GROMACS fork (release-2023 branch)
!git clone --depth 1 -b release-2023 https://github.com/multi-ego/gromacs.git gromacs_src 2>/dev/null

# Configure — CUDA is picked up automatically if a GPU runtime is active
!cmake -S gromacs_src -B gromacs_src/build -G Ninja \
       -DBUILD_TESTING=OFF \
       -DGMX_BUILD_OWN_FFTW=ON \
       -DCMAKE_INSTALL_PREFIX=/usr/local/gromacs_mego \
       > /dev/null 2>&1

# Build and install (~20 min)
!ninja -C gromacs_src/build -j$(nproc) 2>/dev/null
!ninja -C gromacs_src/build install 2>/dev/null

# Make gmx available on PATH
subprocess.run(["ln", "-sf", "/usr/local/gromacs_mego/bin/gmx", "/usr/local/bin/gmx"], check=True)
!gmx --version | head -5

## 1 · Install multi-eGO

Two Colab-specific version constraints are handled automatically by the cell below:

- **numpy**: multi-eGO requires ≥ 2.2.2; Colab ships an older version. Upgrading numpy
  requires a **runtime restart** — the cell will prompt you if this is needed.
  Re-run it after restarting and execution will continue normally.

- **pandas**: without an explicit upper bound, pip resolves to pandas 3.x, which
  breaks several pre-installed Colab packages (`google-colab`, `cudf`, `gradio`, …).
  The cell pins pandas to `>=2.2.3,<3.0.0` before installing multi-eGO so that
  pip's resolver never reaches 3.x.

All other dependencies (`ParmEd`, `scipy`, `tables`, etc.) are installed by
`pip install -e multi-ego` and do not require special handling.

In [ ]:
import sys, subprocess, os

# ── 1. Check / upgrade numpy (multi-eGO requires ≥2.2.2, <2.3) ───────────────
# Colab ships an older numpy; upgrading it requires a kernel restart.
import numpy as _np
_cur_np = tuple(int(x) for x in _np.__version__.split(".")[:3])

if _cur_np < (2, 2, 2):
    print(f"Upgrading numpy {_np.__version__} → >=2.2.2 …")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "numpy>=2.2.2,<2.3"])
    print()
    print("━" * 60)
    print("⚠  numpy was upgraded — a runtime restart is required.")
    print("   Go to  Runtime → Restart runtime  (Ctrl+M .)")
    print("   then re-run this cell to continue.")
    print("━" * 60)
    raise SystemExit("Restart required before proceeding.")

print(f"numpy {_np.__version__} ✓")

# ── 2. Pin pandas to <3.0.0 (Colab packages require pandas<3.0) ───────────────
# multi-eGO requires pandas>=2.2.3; without an upper-bound pip resolves to 3.x,
# which breaks google-colab, cudf, gradio, and several other pre-installed libs.
import pandas as _pd
_cur_pd = tuple(int(x) for x in _pd.__version__.split(".")[:3])

if _cur_pd < (2, 2, 3) or _cur_pd >= (3, 0, 0):
    print(f"Pinning pandas {_pd.__version__} → >=2.2.3,<3.0.0 …")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "pandas>=2.2.3,<3.0.0"])
    import importlib, pandas as _pd2
    print(f"pandas {_pd2.__version__} ✓")
else:
    print(f"pandas {_pd.__version__} ✓")

# ── 3. Clone and install multi-eGO (remaining deps installed automatically) ───
# numpy and pandas are already pinned above, so pip will not upgrade them.
!git clone --depth 1 https://github.com/multi-ego/multi-ego.git > /dev/null 2>&1
!pip install -q -e multi-ego

# ── 4. Set GMXLIB so GROMACS finds the multi-eGO force-field files ────────────
os.environ["GMXLIB"] = os.path.abspath("multi-ego")
print("GMXLIB =", os.environ["GMXLIB"])

## 2 · Upload your input file

Upload a **PDB** or **GRO** file for the protein you want to simulate.
Both formats are accepted by `gmx pdb2gmx`.
The file should contain only protein atoms — remove water, ions, and ligands beforehand
unless they are part of the system you are parameterising.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

upload = widgets.FileUpload(accept='.pdb,.gro', multiple=False, description='Upload PDB/GRO')
display(upload)

In [ ]:
import os

if not upload.value:
    raise RuntimeError("No file uploaded — run the cell above and upload a PDB first.")

file_info  = list(upload.value.values())[0]
filename   = file_info['metadata']['name']

with open(filename, "wb") as f:
    f.write(file_info['content'])

# Derive a system name from the filename (used as the mego --system argument)
SYSTEM = os.path.splitext(os.path.basename(filename))[0]
print(f"Saved    : {filename}")
print(f"System   : {SYSTEM}")

## 3 · Generate AMBER topology  (`gmx pdb2gmx`)

`pdb2gmx` reads the uploaded PDB or GRO file and produces `topol.top` and `processed.gro`
using the **multi-eGO-basic** force field (option 1 in the GMXLIB).  No water model is
added because the mg simulation is run without explicit solvent.

In [ ]:
# Select force field 1 (multi-ego-basic) and water model 1 (none)
!printf "1\n1\n" | gmx pdb2gmx -f {filename} -o processed.gro -water none 2>&1 | tail -15

## 4 · Set the simulation box  (`gmx editconf`)

Creates a cubic periodic box centred on the molecule.  The padding (`-d`, in nm)
ensures the protein does not interact with its periodic image.

In [ ]:
BOX_PADDING = 5.0   # nm — increase for larger proteins

!gmx editconf -f processed.gro -o boxed.gro -c -bt cubic -d {BOX_PADDING} 2>&1 | tail -8

## 5 · Generate the mg force field  (`mego --egos mg`)

`mego --egos mg` reads the AMBER topology and writes the molten-globule (mg)
force-field parameters:

- `topol_mego.top` — GROMACS topology with mg interactions
- `ffnonbonded.itp` — C6/C12 non-bonded parameters (referenced by the topology)

Both files are copied to the working directory.

In [ ]:
import subprocess, shutil, os

# Prepare the mego inputs directory
inputs_dir = os.path.abspath("multi-ego/inputs")
system_dir = os.path.join(inputs_dir, SYSTEM)
os.makedirs(system_dir, exist_ok=True)
shutil.copy("topol.top", os.path.join(system_dir, "topol.top"))

# Run mego
result = subprocess.run(
    ["mego", "--system", SYSTEM, "--egos", "mg", "--inputs_dir", inputs_dir],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
    raise RuntimeError("mego failed — see output above")

# Copy output files to the working directory
outputs_dir = os.path.join("multi-ego", "outputs", SYSTEM)
shutil.copy(os.path.join(outputs_dir, "topol_mego.top"), "topol_mego.top")
shutil.copy(os.path.join(outputs_dir, "ffnonbonded.itp"), "ffnonbonded.itp")
print("\nFiles ready: topol_mego.top  ffnonbonded.itp ✓")

## 6 · Simulation parameters

Edit the values in this cell before running it.  The MDP files for energy minimisation
and the production run are generated automatically from the multi-eGO templates.

In [ ]:
# ── Adjust these values ────────────────────────────────────────────────────────
TEMPERATURE_K = 300        # simulation temperature (K)
NSTEPS        = 1_000_000  # production steps  (1 M × 5 fs = 5 ns)
# ───────────────────────────────────────────────────────────────────────────────

def fill_mdp(template_path, output_path, subs):
    with open(template_path) as f:
        content = f.read()
    for key, val in subs.items():
        content = content.replace(key, str(val))
    with open(output_path, "w") as f:
        f.write(content)

fill_mdp("multi-ego/mdps/ff_em.mdp",  "em.mdp",     {})
fill_mdp("multi-ego/mdps/ff_aa.mdp",  "ff_aa.mdp",
         {"_TEMP_": TEMPERATURE_K, "_SET_": NSTEPS})

print(f"em.mdp    → steepest-descent energy minimisation")
print(f"ff_aa.mdp → {NSTEPS:,} steps at {TEMPERATURE_K} K  ({NSTEPS * 0.005 / 1000:.1f} ns)")

## 7 · Energy minimisation  (`gmx mdrun`)

Steepest-descent minimisation with the mg force field.  Removes steric clashes in
the starting structure before dynamics begin.

In [ ]:
!gmx grompp -f em.mdp -c boxed.gro -p topol_mego.top -o em.tpr -maxwarn 1 2>&1 | tail -5
!gmx mdrun -v -deffnm em -ntmpi 1 2>&1 | tail -10
print("\nMinimisation complete ✓  →  em.gro")

## 8 · NVT production run  (`gmx mdrun`)

Langevin dynamics with the mg force field.  GPU is used automatically when a GPU
runtime is active.  The run produces:

- `run.xtc` — trajectory (needed by `cmdata`)
- `run.tpr` — run-input file (needed by `cmdata`)
- `run.edr` — energy file (used for analysis below)

In [ ]:
!gmx grompp -f ff_aa.mdp -c em.gro -p topol_mego.top -o run.tpr -maxwarn 1 2>&1 | tail -5
!gmx mdrun -v -deffnm run -ntmpi 1 2>&1 | tail -15
print("\nProduction run complete ✓  →  run.xtc  run.tpr  run.edr")

## 9 · Energy analysis

Extracts potential energy and temperature from the GROMACS energy file and plots them.
Both quantities should be stable by the end of the run.

In [ ]:
import subprocess
import numpy as np
import matplotlib.pyplot as plt

# Extract potential energy and temperature from the energy file
subprocess.run(
    "printf 'Potential\\nTemperature\\n0\\n' | gmx energy -f run.edr -o energy.xvg",
    shell=True, capture_output=True
)

# Parse the xvg file (lines starting with # or @ are metadata)
time_ns, potential, temperature = [], [], []
with open("energy.xvg") as f:
    for line in f:
        if line.startswith(('#', '@')):
            continue
        cols = line.split()
        if len(cols) >= 3:
            time_ns.append(float(cols[0]) / 1000)   # ps → ns
            potential.append(float(cols[1]))
            temperature.append(float(cols[2]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(time_ns, potential, color='steelblue', lw=0.8)
ax1.set_xlabel('Time (ns)')
ax1.set_ylabel('Potential energy (kJ/mol)')
ax1.set_title('Potential Energy')
ax1.grid(alpha=0.3)

ax2.plot(time_ns, temperature, color='tomato', lw=0.8)
ax2.axhline(TEMPERATURE_K, color='k', ls='--', lw=1, label=f'{TEMPERATURE_K} K target')
ax2.set_xlabel('Time (ns)')
ax2.set_ylabel('Temperature (K)')
ax2.set_title('Temperature')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('energy_plot.png', dpi=150)
plt.show()

print(f"Mean temperature : {np.mean(temperature):.1f} ± {np.std(temperature):.1f} K")
print(f"Mean pot. energy : {np.mean(potential):.1f} kJ/mol")

## 10 · Download output files

Download the files you need for the next steps of the multi-eGO workflow.

| File | Used for |
|------|----------|
| `run.xtc` | contact histogram extraction with `cmdata` |
| `run.tpr` | required by `cmdata` alongside the trajectory |
| `run.edr` | GROMACS energy file (optional, for further analysis) |
| `energy_plot.png` | visual sanity check |

In [ ]:
from google.colab import files as colab_files
import os

for f in ["run.xtc", "run.tpr", "run.edr", "energy_plot.png"]:
    if os.path.exists(f):
        print(f"Downloading {f}…")
        colab_files.download(f)
    else:
        print(f"  (skipping {f} — not found)")

## Next steps

With `run.xtc` and `run.tpr` in hand, continue the multi-eGO workflow locally:

1. **Extract contact histograms** with `cmdata`:
   ```bash
   cmdata -f run.xtc -s run.tpr
   ```

2. **Convert histograms to contact matrices** with `make_mat`:
   ```bash
   python tools/make_mat/make_mat.py --histo histo/ ...
   ```

3. **Generate the production force field** with `mego`:
   ```bash
   mego --config inputs/SYSTEM/config.yml
   ```

See the [multi-eGO documentation](https://github.com/multi-ego/multi-eGO) for the full workflow.